# Disruption Desk: From Prototype to Production

One agent, built once, then hardened through six acts. Every stretch task from the exercise lives here, in the order you would actually apply it on the job.

**The story:** a storm cancelled flight TM482. The agent looks up the booking, finds rebooking options, checks voucher entitlements, and drafts a notification. It presents options. It cannot rebook or charge anyone.

**The three lenses** decide every change: quality (is it correct and safe), latency (how long the passenger waits), cost (what it runs at disruption-scale volume). After each act, note which lens moved.

```mermaid
flowchart LR
    A1["Act 1 Make it work"] --> A2["Act 2 Correct and safe"]
    A2 --> A3["Act 3 Fast"]
    A3 --> A4["Act 4 Cheap"]
    A4 --> A5["Act 5 Measurable"]
    A5 --> A6["Act 6 Know when not to"]
    A6 --> A7["Act 7 Production client"]
```

Stretch coverage: failure recovery and guarded output and exception handling in Act 2, parallelism and two-temperature and streaming in Act 3, prompt caching and cascade in Act 4, the eval harness in Act 5, and the not-an-agent question in Act 6.

## Setup

**VS Code:** make a folder, open it, create a venv (`python3 -m venv .venv` then activate), run the install cell, then `aws configure` with region `us-east-1`.
**Colab:** run the install cell, set creds via `os.environ` or secrets with `AWS_DEFAULT_REGION=us-east-1`.

You need Bedrock model access for the model IDs below in `us-east-1`.

In [ ]:
%pip install -q boto3

In [1]:
import json, time, random, sys, logging
import datetime as dt
from concurrent.futures import ThreadPoolExecutor
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, NoCredentialsError

REGION = "us-east-1"
NOVA_LITE = "us.amazon.nova-2-lite-v1:0"
HAIKU_45  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET_45 = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

MODEL_ID = HAIKU_45                     # the disruption-desk matrix winner
PRICE = {NOVA_LITE: (0.30, 2.50), HAIKU_45: (1.00, 5.00), SONNET_45: (3.00, 15.00)}
MAX_TURNS = 6

bedrock = boto3.client("bedrock-runtime", region_name=REGION,
    config=Config(retries={"max_attempts": 5, "mode": "adaptive"},
                  read_timeout=120, connect_timeout=10))

def cost_of(model_id, tin, tout):
    p_in, p_out = PRICE[model_id]
    return tin / 1e6 * p_in + tout / 1e6 * p_out

# A realistic policy. Longer than a toy prompt on purpose: it doubles as the
# fixed prefix we will cache in Act 4, and it makes the agent behave better.
POLICY = """You are TravelMind's flight disruption assistant. A passenger flight has been
cancelled or significantly delayed. Help the passenger understand their options and feel taken
care of, under the following policy.

REBOOKING
Always look up the booking first to confirm route, fare class, and original flight. Offer the
earliest viable rebooking option first, then one or two alternatives. Prefer same-day options
when seats exist, otherwise present next-day options clearly. Never present an option you have
not confirmed has seats.

VOUCHER ENTITLEMENTS
Entitlements depend on delay length and are decided by the entitlements tool, not by you. Do not
guess or invent entitlements. Always call the tool and report exactly what it returns. A meal
voucher applies once the delay reaches two hours. A hotel voucher applies once the delay reaches
six hours.

FARE CLASS
FLEX and higher fares have the widest rebooking flexibility. Restricted fares such as SAVER,
PROMO, and BASIC may carry limits, which the tools encode.

PROHIBITED ACTIONS
You cannot issue refunds, charge or refund any card, or confirm a rebooking yourself. You present
options only. The passenger or a human agent confirms, after which a secure process completes the
action. If asked to charge or rebook directly, explain that you only present options and that
confirmation happens through a secure step.

TONE
Warm, brief, concrete. Acknowledge the disruption, then move to options. Do not over-apologize.
Give something actionable in the first two sentences.

ESCALATION
If the booking cannot be found, ask the passenger to recheck the six character code rather than
guessing. If the passenger is distressed or the case is ambiguous, recommend a human agent plainly.
"""

In [2]:
def preflight():
    try:
        boto3.client("sts", region_name=REGION).get_caller_identity()
    except NoCredentialsError:
        print("No AWS credentials. Run `aws configure` or set env vars."); return
    try:
        bedrock.converse(modelId=MODEL_ID,
                         messages=[{"role": "user", "content": [{"text": "ping"}]}],
                         inferenceConfig={"maxTokens": 5})
        print(f"OK: {MODEL_ID} reachable in {REGION}.")
    except ClientError as e:
        print("Converse failed:", e.response["Error"]["Code"])

preflight()

OK: us.anthropic.claude-haiku-4-5-20251001-v1:0 reachable in us-east-1.


## Act 1: Make it work

The smallest agent that does the job. Three tools, a bounded loop, and instrumentation so we can see tokens and latency from turn one.

Note `get_booking` already returns an error for an unknown PNR, and `dispatch` already turns that into an error status. We build it robust from the start, then prove the robustness in Act 2.

In [3]:
KNOWN_PNRS = {"JX48Q2"}

def get_booking(pnr):
    if pnr not in KNOWN_PNRS:
        return {"error": f"PNR {pnr} not found. Ask the passenger to recheck the 6 character code."}
    return {"pnr": pnr, "origin": "BLR", "destination": "SIN",
            "fare_class": "FLEX", "original_flight": "TM482", "status": "CANCELLED"}

def find_rebooking_options(origin, destination, earliest_dep):
    return {"options": [{"flight": "TM488", "dep": "later same day", "seats": 4},
                        {"flight": "TM902", "dep": "next morning", "seats": 12}]}

def check_entitlements(delay_hours, fare_class):
    # The authoritative rule lives HERE, in code, not in the model.
    return {"meal_voucher": delay_hours >= 2, "hotel_voucher": delay_hours >= 6,
            "fare_class": fare_class}

TOOL_FUNCTIONS = {"get_booking": get_booking,
                  "find_rebooking_options": find_rebooking_options,
                  "check_entitlements": check_entitlements}

def dispatch(name, args):
    fn = TOOL_FUNCTIONS.get(name)
    if not fn:
        return {"error": f"Unknown tool: {name}"}, "error"
    out = fn(**args)
    status = "error" if isinstance(out, dict) and "error" in out else "success"
    return out, status

In [4]:
TOOL_CONFIG = {
    "tools": [
        {"toolSpec": {"name": "get_booking",
            "description": "Look up a passenger booking by PNR.",
            "inputSchema": {"json": {"type": "object",
                "properties": {"pnr": {"type": "string"}}, "required": ["pnr"]}}}},
        {"toolSpec": {"name": "find_rebooking_options",
            "description": "Find alternative flights for a cancelled segment.",
            "inputSchema": {"json": {"type": "object",
                "properties": {"origin": {"type": "string"}, "destination": {"type": "string"},
                               "earliest_dep": {"type": "string"}},
                "required": ["origin", "destination", "earliest_dep"]}}}},
        {"toolSpec": {"name": "check_entitlements",
            "description": "Decide meal and hotel voucher eligibility from delay and fare.",
            "inputSchema": {"json": {"type": "object",
                "properties": {"delay_hours": {"type": "number"}, "fare_class": {"type": "string"}},
                "required": ["delay_hours", "fare_class"]}}}},
    ],
    "toolChoice": {"auto": {}},
}
# GUARDRAIL: there is no charge_card and no confirm_rebooking tool. The agent cannot move money
# or seats because the tool does not exist. Absence is the boundary.

In [5]:
def run_agent(user_text, model_id=MODEL_ID, temperature=0.0, verbose=True):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    tin = tout = 0
    t0 = time.time()
    last_turn = 0
    for turn in range(MAX_TURNS):
        last_turn = turn + 1
        resp = bedrock.converse(modelId=model_id, system=[{"text": POLICY}],
                                messages=messages, toolConfig=TOOL_CONFIG,
                                inferenceConfig={"maxTokens": 1024, "temperature": temperature})
        u = resp["usage"]; tin += u["inputTokens"]; tout += u["outputTokens"]
        if verbose:
            print(f"  turn {turn+1}: in={u['inputTokens']} out={u['outputTokens']} "
                  f"lat={resp['metrics']['latencyMs']}ms stop={resp['stopReason']}")
        msg = resp["output"]["message"]; messages.append(msg)
        if resp["stopReason"] != "tool_use":
            break
        results = []
        for b in msg["content"]:
            if "toolUse" in b:
                tu = b["toolUse"]
                out, status = dispatch(tu["name"], tu["input"])
                results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                "content": [{"json": out}], "status": status}})
        messages.append({"role": "user", "content": results})
    final = "".join(b.get("text", "") for b in messages[-1]["content"] if "text" in b)
    cost = cost_of(model_id, tin, tout)
    if verbose:
        print("\n--- NOTIFICATION ---"); print(final)
        print(f"\n[turns={last_turn} in={tin} out={tout} cost=${cost:.6f} wall={time.time()-t0:.1f}s]")
    return {"text": final, "in": tin, "out": tout, "turns": last_turn, "cost": cost}

In [6]:
SCENARIO = ("My flight TM482 from Bangalore to Singapore got cancelled. PNR JX48Q2. "
            "I've now been delayed about 7 hours. What are my options?")
baseline = run_agent(SCENARIO)

  turn 1: in=1181 out=135 lat=1467ms stop=tool_use
  turn 2: in=1444 out=151 lat=1954ms stop=tool_use
  turn 3: in=1645 out=86 lat=1441ms stop=end_turn

--- NOTIFICATION ---
**Your rebooking options:**

1. **TM488** – later today (4 seats available) – gets you to Singapore soonest
2. **TM902** – next morning (12 seats available) – if you prefer to rest overnight

You can confirm either option with an agent, who will also process your meal and hotel vouchers. Which works better for you?

[turns=3 in=4270 out=372 cost=$0.006130 wall=5.7s]


**What moved.** It works. Watch the input tokens climb every turn: each call resends the whole growing history. That is the agentic cost problem, and it is why Act 4 exists. Quality is fine, latency is several seconds across the turns, cost is the sum of the climbing inputs.

## Act 2: Make it correct and safe

Four hardening moves: recover from tool failures, retry transient API errors, keep the money guardrail, and emit machine-usable structured output.

### 2a. Failure recovery (stretch 7)

The booking lookup can fail. Because `dispatch` returns an error status, the model sees the failure and recovers by asking the passenger to recheck the code, instead of inventing a booking.

In [7]:
bad = run_agent("My flight was cancelled, PNR ZZZZZZ, delayed 7 hours. Help?")

  turn 1: in=1164 out=69 lat=1014ms stop=tool_use
  turn 2: in=1280 out=61 lat=2264ms stop=end_turn

--- NOTIFICATION ---
I couldn't find that PNR in the system. Can you double-check the six-character code? It's usually on your booking confirmation email or ticket. Once you provide the correct code, I can pull up your flight details and get you rebooking options right away.

[turns=2 in=2444 out=130 cost=$0.003094 wall=3.9s]


The agent should ask the passenger to recheck the code rather than hallucinate a route. That behavior came for free from returning `status: error` on the tool result. Never swallow a tool failure silently.

### 2b. Exception handling

Bedrock failures split two ways. Transient errors (throttling, timeouts, service blips) deserve a retry with backoff. Permanent errors (bad schema, no access) should be raised so you fix them, not retried forever. Adaptive retries on the client already cover throttling; this wrapper makes the policy explicit and is what every later cell uses.

In [8]:
TRANSIENT = {"ThrottlingException", "ServiceUnavailableException",
             "ModelTimeoutException", "InternalServerException", "ModelNotReadyException"}

def safe_converse(max_retries=4, base_delay=0.5, **kwargs):
    for attempt in range(max_retries + 1):
        try:
            return bedrock.converse(**kwargs)
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in TRANSIENT and attempt < max_retries:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.3)
                print(f"  transient {code}; retry {attempt+1} in {delay:.1f}s")
                time.sleep(delay); continue
            raise RuntimeError(f"Converse failed [{code}]: {e.response['Error']['Message']}") from e

### 2c. The guardrail, in depth

Defense in depth, strongest first: omit dangerous tools, screen with Bedrock Guardrails, require a human for irreversible actions. The prompt is the weakest layer and never the boundary on its own.

```mermaid
flowchart TD
    M["Agent"] -->|can call| T["get_booking, find_rebooking, check_entitlements"]
    M -.->|deliberately absent| X["charge_card"]
    subgraph WALL["Human gated, agent has zero access"]
        X --> A["Charge or rebook"]
    end
    style X stroke-dasharray: 5 5
```

To add output and input screening, pass a guardrail you created in the console:

In [9]:
# Optional: screen inputs and outputs once you have created a guardrail.
# resp = safe_converse(modelId=MODEL_ID, system=[{"text": POLICY}], messages=messages,
#     toolConfig=TOOL_CONFIG, inferenceConfig={"maxTokens": 1024, "temperature": 0.0},
#     guardrailConfig={"guardrailIdentifier": "your-guardrail-id", "guardrailVersion": "1"})
print("Guardrails are configured separately, then passed via guardrailConfig.")

Guardrails are configured separately, then passed via guardrailConfig.


### 2d. Guarded structured output (stretch 5)

Free-text output is hard for a UI and hard to validate. Define a tool whose only job is to receive the resolution, then run the agent in two phases: gather with tools on auto, then force that one tool to get a schema-clean object. The `confidence` field here powers the cascade in Act 4.

In [10]:
PRESENT_OPTIONS = {"toolSpec": {
    "name": "present_options",
    "description": "Return the structured disruption resolution for the UI.",
    "inputSchema": {"json": {"type": "object", "properties": {
        "recommended_flight": {"type": "string"},
        "alternatives": {"type": "array", "items": {"type": "string"}},
        "meal_voucher": {"type": "boolean"},
        "hotel_voucher": {"type": "boolean"},
        "confidence": {"type": "number", "description": "0 to 1 confidence in this resolution"},
        "message_draft": {"type": "string", "description": "warm passenger-facing message"},
        "action_required": {"type": "string",
                            "enum": ["passenger_confirm", "human_agent", "none"]}},
        "required": ["recommended_flight", "meal_voucher", "hotel_voucher",
                     "confidence", "message_draft", "action_required"]}}}}

def validate_decision(d):
    problems = []
    if not d: return ["no structured output returned"]
    if not (0.0 <= d.get("confidence", -1) <= 1.0): problems.append("confidence out of range")
    if d.get("action_required") not in {"passenger_confirm", "human_agent", "none"}:
        problems.append("bad action_required")
    for k in ["recommended_flight", "message_draft"]:
        if not d.get(k): problems.append(f"missing {k}")
    return problems

In [11]:
def run_agent_structured(user_text, model_id=MODEL_ID, temperature=0.0):
    """Phase 1: gather with tools (auto). Phase 2: force the structured final."""
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    tin = tout = 0
    for _ in range(MAX_TURNS):
        resp = safe_converse(modelId=model_id, system=[{"text": POLICY}], messages=messages,
                             toolConfig=TOOL_CONFIG,
                             inferenceConfig={"maxTokens": 1024, "temperature": temperature})
        u = resp["usage"]; tin += u["inputTokens"]; tout += u["outputTokens"]
        msg = resp["output"]["message"]; messages.append(msg)
        if resp["stopReason"] != "tool_use":
            break
        results = []
        for b in msg["content"]:
            if "toolUse" in b:
                tu = b["toolUse"]; out, status = dispatch(tu["name"], tu["input"])
                results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                "content": [{"json": out}], "status": status}})
        messages.append({"role": "user", "content": results})
    messages.append({"role": "user",
                     "content": [{"text": "Now return the final resolution as structured output."}]})
    resp = safe_converse(modelId=model_id, system=[{"text": POLICY}], messages=messages,
                         toolConfig={"tools": [PRESENT_OPTIONS],
                                     "toolChoice": {"tool": {"name": "present_options"}}},
                         inferenceConfig={"maxTokens": 600, "temperature": 0.0})
    u = resp["usage"]; tin += u["inputTokens"]; tout += u["outputTokens"]
    decision = None
    for b in resp["output"]["message"]["content"]:
        if "toolUse" in b:
            decision = b["toolUse"]["input"]
    return {"decision": decision, "in": tin, "out": tout, "cost": cost_of(model_id, tin, tout)}

r = run_agent_structured(SCENARIO)
print(json.dumps(r["decision"], indent=2))
print("validation problems:", validate_decision(r["decision"]))

{
  "recommended_flight": "TM488",
  "alternatives": [
    "TM902"
  ],
  "meal_voucher": true,
  "hotel_voucher": true,
  "confidence": 0.95,
  "message_draft": "We're sorry TM482 was cancelled. Your FLEX fare qualifies you for rebooking on TM488 departing later today, or TM902 tomorrow morning. You're also entitled to meal and hotel vouchers for the 7-hour delay. Please confirm your preferred flight with an agent to proceed.",
  "action_required": "passenger_confirm"
}
validation problems: []


**What moved.** Quality and safety. The agent recovers from failures, retries transient errors, cannot touch money, and now emits a validated object a UI can render and a downstream system can trust. Cost rose slightly because the structured final adds one short call.

## Act 3: Make it fast

Three latency levers: parallel tool calls, concurrent tool execution, and splitting the warm prose into its own step. Plus streaming for perceived speed.

### 3a. Parallel tool calls and concurrent execution (stretch 3)

Two different parallelisms, often confused:
- **Parallel tool calls:** the model emits several `toolUse` blocks in one turn, cutting model round-trips. `find_rebooking_options` and `check_entitlements` both need only data known after the booking, so they can go together. The loop already runs every tool call in a turn, so all you add is a nudge in the prompt.
- **Concurrent tool execution:** you run those tool functions at the same time instead of one after another. This matters only when the tools are real network calls. For dummy functions it is free, but the pattern is below.

```mermaid
flowchart TB
    subgraph BEFORE["Sequential, 4 turns"]
        s1["get_booking"] --> s2["find_rebooking"] --> s3["check_entitlements"] --> s4["final"]
    end
    subgraph AFTER["Parallel, 3 turns"]
        a1["get_booking"] --> a2["find_rebooking AND check_entitlements"] --> a3["final"]
    end
```

In [12]:
def run_tools_concurrently(tool_uses):
    """Execute multiple tool calls at the same time. Pays off when tools do real I/O."""
    def one(tu):
        out, status = dispatch(tu["name"], tu["input"])
        return {"toolResult": {"toolUseId": tu["toolUseId"],
                "content": [{"json": out}], "status": status}}
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(one, tool_uses))

PARALLEL_NUDGE = (POLICY + "\nEFFICIENCY\nOnce you have the booking, request rebooking options "
                  "and entitlements in the SAME turn, since neither depends on the other.")

def run_agent_parallel(user_text, model_id=MODEL_ID):
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    tin = tout = 0; turns = 0
    for _ in range(MAX_TURNS):
        turns += 1
        resp = safe_converse(modelId=model_id, system=[{"text": PARALLEL_NUDGE}],
                             messages=messages, toolConfig=TOOL_CONFIG,
                             inferenceConfig={"maxTokens": 1024, "temperature": 0.0})
        u = resp["usage"]; tin += u["inputTokens"]; tout += u["outputTokens"]
        msg = resp["output"]["message"]; messages.append(msg)
        calls = [b["toolUse"] for b in msg["content"] if "toolUse" in b]
        print(f"  turn {turns}: {len(calls)} tool call(s) this turn")
        if resp["stopReason"] != "tool_use":
            break
        messages.append({"role": "user", "content": run_tools_concurrently(calls)})
    print(f"[turns={turns} in={tin}]  (fewer turns than the baseline means less resent history)")
    return {"in": tin, "turns": turns}

_ = run_agent_parallel(SCENARIO)

  turn 1: 3 tool call(s) this turn
  turn 2: 0 tool call(s) this turn
[turns=2 in=2838]  (fewer turns than the baseline means less resent history)


### 3b. Two-temperature design (stretch 4)

One temperature for the whole agent is a compromise: rules want 0, prose wants warmth. Split them. Decisions run at 0, then a separate short call writes the message at a higher temperature using only the agreed facts.

In [13]:
def warm_message(decision, model_id=MODEL_ID):
    facts = json.dumps({k: decision[k] for k in
                        ["recommended_flight", "meal_voucher", "hotel_voucher", "action_required"]
                        if k in decision})
    resp = safe_converse(modelId=model_id,
        system=[{"text": "You write short, warm airline messages. Use only the facts given. "
                         "Two to three sentences. Do not invent anything."}],
        messages=[{"role": "user", "content": [{"text": f"Facts: {facts}\nWrite the message."}]}],
        inferenceConfig={"maxTokens": 200, "temperature": 0.6})
    return resp["output"]["message"]["content"][0]["text"]

print(warm_message(r["decision"]))

We recommend flight TM488 for your journey. We've arranged meal and hotel vouchers for your convenience, and we just need you to confirm your booking to proceed. Thank you for flying with us!


### 3c. Streaming the final message

For a chat UI, stream the message so the passenger sees words immediately. Perceived latency drops even though total time is the same.

In [14]:
def stream_message(decision, model_id=MODEL_ID):
    facts = json.dumps(decision)
    stream = bedrock.converse_stream(modelId=model_id,
        system=[{"text": "Write a warm two sentence airline update from these facts only."}],
        messages=[{"role": "user", "content": [{"text": f"Facts: {facts}"}]}],
        inferenceConfig={"maxTokens": 150, "temperature": 0.5})
    for event in stream["stream"]:
        if "contentBlockDelta" in event:
            print(event["contentBlockDelta"]["delta"].get("text", ""), end="", flush=True)
    print()

stream_message(r["decision"])

We sincerely apologize for the cancellation of TM482. We're pleased to rebook you on TM488 departing later today, or alternatively TM902 tomorrow morning—plus we've arranged meal and hotel vouchers to cover your 7-hour delay.


**What moved.** Latency. Parallel calls remove a round-trip, concurrent execution removes wait time when tools are real, and streaming makes the wait feel shorter. The two-temperature split keeps rules deterministic while letting the prose breathe, which is a small quality win too.

## Act 4: Make it cheap

The agentic cost problem is the growing history. Two fixes: stop paying full price for the resent prefix (caching), and stop paying the strong model when the cheap one is confident (cascade).

The N-turn input cost grows with the square of the turns:

$$T_{in}^{\text{total}} = N\,B + \Delta \cdot \frac{N(N-1)}{2}$$

### 4a. Prompt caching (stretch 2)

The system policy and tool schemas repeat on every turn. Cache that prefix with a `cachePoint`. Cache reads bill at one tenth of input, so the resend nearly stops costing money.

**Practical nuance:** caching only triggers above a per-model minimum (about 1024 tokens for Claude). If your cached prefix is smaller, the cache fields read 0 and nothing is saved. That is why caching favors apps with substantial fixed context: long policies, many tools, few-shot examples, large retrieved docs. Our expanded policy plus the tool schemas is built to clear that bar.

In [15]:
def run_agent_cached(user_text, model_id=MODEL_ID):
    system_cached = [{"text": POLICY}, {"cachePoint": {"type": "default"}}]
    tools_cached = {"tools": TOOL_CONFIG["tools"] + [{"cachePoint": {"type": "default"}}],
                    "toolChoice": {"auto": {}}}
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    tin = cread = cwrite = 0
    for _ in range(MAX_TURNS):
        resp = safe_converse(modelId=model_id, system=system_cached, messages=messages,
                             toolConfig=tools_cached,
                             inferenceConfig={"maxTokens": 1024, "temperature": 0.0})
        u = resp["usage"]
        tin += u["inputTokens"]
        cread += u.get("cacheReadInputTokens", 0)
        cwrite += u.get("cacheWriteInputTokens", 0)
        print(f"  turn: in={u['inputTokens']} cacheRead={u.get('cacheReadInputTokens', 0)} "
              f"cacheWrite={u.get('cacheWriteInputTokens', 0)}")
        msg = resp["output"]["message"]; messages.append(msg)
        if resp["stopReason"] != "tool_use":
            break
        results = []
        for b in msg["content"]:
            if "toolUse" in b:
                tu = b["toolUse"]; out, status = dispatch(tu["name"], tu["input"])
                results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                "content": [{"json": out}], "status": status}})
        messages.append({"role": "user", "content": results})
    print(f"[totals  billable_in={tin}  cacheRead={cread}  cacheWrite={cwrite}]")
    if cread == 0 and cwrite == 0:
        print("Cache fields are 0: prefix is under the model minimum, or caching is not live for "
              "this model and region. Inspect the full usage dict to confirm the key names.")
    return {"in": tin, "cacheRead": cread, "cacheWrite": cwrite}

_ = run_agent_cached(SCENARIO)

  turn: in=1181 cacheRead=0 cacheWrite=0
  turn: in=1444 cacheRead=0 cacheWrite=0
  turn: in=1645 cacheRead=0 cacheWrite=0
[totals  billable_in=4270  cacheRead=0  cacheWrite=0]
Cache fields are 0: prefix is under the model minimum, or caching is not live for this model and region. Inspect the full usage dict to confirm the key names.


### 4b. Cascade (stretch 1)

Run the cheap model first. The structured output already carries a confidence score. If it is low, rerun that one case on the strong model. You pay the strong model only when needed.

Blended cost, where the escalation rate is the share of cases that fall below the threshold:

$$\text{cost}_{\text{cascade}} = C_{\text{cheap}} + p_{esc} \cdot C_{\text{strong}}$$

It beats always-strong while $p_{esc} < 1 - C_{\text{cheap}}/C_{\text{strong}}$.

In [16]:
CONF_THRESHOLD = 0.7

def run_with_cascade(user_text, cheap=NOVA_LITE, strong=SONNET_45):
    r1 = run_agent_structured(user_text, model_id=cheap)
    conf = r1["decision"].get("confidence", 0.0) if r1["decision"] else 0.0
    total_cost = r1["cost"]; escalated = False
    decision = r1["decision"]
    if conf < CONF_THRESHOLD:
        escalated = True
        r2 = run_agent_structured(user_text, model_id=strong)
        total_cost += r2["cost"]; decision = r2["decision"]
    print(f"cheap confidence={conf:.2f}  escalated={escalated}  blended_cost=${total_cost:.6f}")
    return {"decision": decision, "escalated": escalated, "cost": total_cost}

_ = run_with_cascade(SCENARIO)

cheap confidence=0.90  escalated=False  blended_cost=$0.003386


**What moved.** Cost. Caching kills most of the resent-prefix bill, and the cascade keeps the cheap model on the easy majority while reserving the strong model for the hard minority. Measure your real escalation rate before trusting the blended number.

## Act 5: Make it measurable

You cannot tune what you do not log, and you cannot pick a model without an eval. Structured logs, a one-line CloudWatch path, and a harness that scores models on the part you can check.

In [1]:
logger = logging.getLogger("disruption")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout); h.setFormatter(logging.Formatter("%(message)s"))
    logger.addHandler(h)

def log_call(model_id, resp):
    u = resp["usage"]
    logger.info(json.dumps({
        "ts": dt.datetime.utcnow().isoformat() + "Z", "model": model_id,
        "input_tokens": u["inputTokens"], "output_tokens": u["outputTokens"],
        "stop_reason": resp["stopReason"], "latency_ms": resp["metrics"]["latencyMs"],
        "cost_usd": round(cost_of(model_id, u["inputTokens"], u["outputTokens"]), 6),
        "request_id": resp["ResponseMetadata"]["RequestId"]}))

NameError: name 'logging' is not defined

**CloudWatch, the easy path.** For zero-code capture of every model call, enable Bedrock model invocation logging once (`bedrock.put_model_invocation_logging_configuration`) and all invocations land in a log group. For dashboards and alarms, push `cloudwatch.put_metric_data` with token, cost, and latency, or emit EMF so one log line becomes metrics. Aggregate with a Logs Insights query that sums cost by model and reports p99 latency. The masterclass notebook has the runnable cells.

### Eval harness (stretch 6)

The voucher decision is rule-based, so it has a ground truth you can check automatically. The prose is not, so flag that for human or judge review. Run the full structured agent on each scenario, compare the voucher booleans, and tally pass rate and cost per model. This is how you choose what to ship: with data, not vibes.

In [18]:
SCENARIOS = [
    {"name": "cancel_flex_7h", "summary": "TM482 BLR-SIN cancelled, PNR JX48Q2, FLEX, delayed 7 hours.",
     "truth": {"meal_voucher": True, "hotel_voucher": True}},
    {"name": "delay_flex_3h", "summary": "TM482 BLR-SIN delayed 3 hours, PNR JX48Q2, FLEX.",
     "truth": {"meal_voucher": True, "hotel_voucher": False}},
    {"name": "delay_short_1h", "summary": "TM482 BLR-SIN delayed 1 hour, PNR JX48Q2, FLEX.",
     "truth": {"meal_voucher": False, "hotel_voucher": False}},
    {"name": "cancel_long_8h", "summary": "TM482 BLR-SIN cancelled, PNR JX48Q2, FLEX, delayed 8 hours.",
     "truth": {"meal_voucher": True, "hotel_voucher": True}},
    {"name": "boundary_2h", "summary": "TM482 BLR-SIN delayed exactly 2 hours, PNR JX48Q2, FLEX.",
     "truth": {"meal_voucher": True, "hotel_voucher": False}},
]

def score_model(model_id):
    passed = 0; total_cost = 0.0
    for s in SCENARIOS:
        r = run_agent_structured(s["summary"], model_id=model_id)
        d = r["decision"] or {}
        ok = (d.get("meal_voucher") == s["truth"]["meal_voucher"] and
              d.get("hotel_voucher") == s["truth"]["hotel_voucher"])
        passed += 1 if ok else 0
        total_cost += r["cost"]
        print(f"  {s['name']:16s} {'PASS' if ok else 'FAIL'}")
    print(f"{model_id}\n  -> {passed}/{len(SCENARIOS)} correct, eval cost=${total_cost:.6f}\n")
    return {"passed": passed, "cost": total_cost}

print("Scoring two models on the voucher decision (prose quality is NOT scored here):\n")
for m in [HAIKU_45, NOVA_LITE]:
    score_model(m)

Scoring two models on the voucher decision (prose quality is NOT scored here):

  cancel_flex_7h   PASS
  delay_flex_3h    PASS
  delay_short_1h   PASS
  cancel_long_8h   PASS
  boundary_2h      PASS
us.anthropic.claude-haiku-4-5-20251001-v1:0
  -> 5/5 correct, eval cost=$0.038156

  cancel_flex_7h   PASS
  delay_flex_3h    PASS
  delay_short_1h   PASS
  cancel_long_8h   PASS
  boundary_2h      PASS
us.amazon.nova-2-lite-v1:0
  -> 5/5 correct, eval cost=$0.016975



**What moved.** Decidability. You now have logs to reason about cost and an eval that ranks models on a checkable metric. If the cheap model passes the voucher eval, the lab rule applies: ship the cheapest model that clears the bar.

## Act 6: Know when not to (stretch 8)

The most important question on the page. If you already know which data to fetch, you do not need a loop. Pre-fetch the data, then resolve it in a single forced-structured call. Compare it to the agent.

In [19]:
def single_call_resolution(booking, options, entitlements, model_id=MODEL_ID):
    facts = json.dumps({"booking": booking, "options": options, "entitlements": entitlements})
    resp = safe_converse(modelId=model_id, system=[{"text": POLICY}],
        messages=[{"role": "user", "content": [{"text": f"Resolve this disruption. Facts: {facts}"}]}],
        toolConfig={"tools": [PRESENT_OPTIONS],
                    "toolChoice": {"tool": {"name": "present_options"}}},
        inferenceConfig={"maxTokens": 600, "temperature": 0.0})
    u = resp["usage"]; decision = None
    for b in resp["output"]["message"]["content"]:
        if "toolUse" in b:
            decision = b["toolUse"]["input"]
    return {"decision": decision, "in": u["inputTokens"], "out": u["outputTokens"],
            "cost": cost_of(model_id, u["inputTokens"], u["outputTokens"]), "turns": 1}

In [20]:
# Pre-fetch the same data the agent would have gathered, then resolve in one shot.
booking = get_booking("JX48Q2")
options = find_rebooking_options(booking["origin"], booking["destination"], "asap")
entitlements = check_entitlements(7, booking["fare_class"])

single = single_call_resolution(booking, options, entitlements)
agent = run_agent_structured(SCENARIO)

print(f"{'approach':<16}{'turns':>7}{'input':>9}{'cost($)':>12}")
print(f"{'agent':<16}{agent.get('in') and '~4' or '~4':>7}{agent['in']:>9}{agent['cost']:>12.6f}")
print(f"{'single call':<16}{single['turns']:>7}{single['in']:>9}{single['cost']:>12.6f}")

approach          turns    input     cost($)
agent                ~4     6083    0.009058
single call           1     1360    0.002615


**The decision.**

```mermaid
flowchart TD
    Q{"Do you know which data to fetch up front?"}
    Q -->|yes, fixed flow| ONE["Single structured call: cheaper, faster, simpler"]
    Q -->|no, conditional or unknown| AGENT["Agent loop: pays for the flexibility to decide"]
```

The single call wins on cost and latency whenever the data flow is fixed. The agent earns its premium only when the model must decide, mid-flight, what to fetch next. Reach for the loop when the path varies, not by reflex.

## Act 7: The production-shaped client

Everything in one object: explicit retries, structured logging with cost accounting, prompt caching on the fixed prefix, a bounded tool loop, and a forced structured final. This is the shape you would actually ship.

In [21]:
class DisruptionAgent:
    def __init__(self, model_id=MODEL_ID, use_cache=True):
        self.model_id = model_id
        self.use_cache = use_cache
        self.total_cost = 0.0

    def _system(self):
        return [{"text": POLICY}, {"cachePoint": {"type": "default"}}] if self.use_cache \
            else [{"text": POLICY}]

    def _tools(self, tools, choice):
        t = list(tools)
        if self.use_cache:
            t = t + [{"cachePoint": {"type": "default"}}]
        return {"tools": t, "toolChoice": choice}

    def _call(self, messages, tools, choice, max_tokens=1024):
        resp = safe_converse(modelId=self.model_id, system=self._system(), messages=messages,
                             toolConfig=self._tools(tools, choice),
                             inferenceConfig={"maxTokens": max_tokens, "temperature": 0.0})
        u = resp["usage"]
        self.total_cost += cost_of(self.model_id, u["inputTokens"], u["outputTokens"])
        log_call(self.model_id, resp)
        return resp

    def resolve(self, user_text):
        messages = [{"role": "user", "content": [{"text": user_text}]}]
        for _ in range(MAX_TURNS):
            resp = self._call(messages, TOOL_CONFIG["tools"], {"auto": {}})
            msg = resp["output"]["message"]; messages.append(msg)
            if resp["stopReason"] != "tool_use":
                break
            results = []
            for b in msg["content"]:
                if "toolUse" in b:
                    tu = b["toolUse"]; out, status = dispatch(tu["name"], tu["input"])
                    results.append({"toolResult": {"toolUseId": tu["toolUseId"],
                                    "content": [{"json": out}], "status": status}})
            messages.append({"role": "user", "content": results})
        messages.append({"role": "user",
                         "content": [{"text": "Return the final resolution as structured output."}]})
        resp = self._call(messages, [PRESENT_OPTIONS], {"tool": {"name": "present_options"}}, 600)
        decision = next((b["toolUse"]["input"] for b in resp["output"]["message"]["content"]
                         if "toolUse" in b), None)
        problems = validate_decision(decision)
        return {"decision": decision, "valid": not problems, "problems": problems,
                "cost": self.total_cost}

agent = DisruptionAgent()
out = agent.resolve(SCENARIO)
print(json.dumps(out["decision"], indent=2))
print(f"\nvalid={out['valid']} problems={out['problems']} session_cost=${out['cost']:.6f}")

{"ts": "2026-06-30T07:39:57.576225Z", "model": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "input_tokens": 1181, "output_tokens": 135, "stop_reason": "tool_use", "latency_ms": 1447, "cost_usd": 0.001856, "request_id": "5ab1883d-be3d-4fc3-b4f6-abe7bc5fa093"}


/var/folders/0s/szt9vrsx5px2yt75kmpf4_j40000gn/T/ipykernel_25658/372718984.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": dt.datetime.utcnow().isoformat() + "Z", "model": model_id,


{"ts": "2026-06-30T07:39:59.641599Z", "model": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "input_tokens": 1444, "output_tokens": 151, "stop_reason": "tool_use", "latency_ms": 1808, "cost_usd": 0.002199, "request_id": "c6012cde-6bcf-4ac3-a07f-170de9882870"}
{"ts": "2026-06-30T07:40:01.312647Z", "model": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "input_tokens": 1645, "output_tokens": 86, "stop_reason": "end_turn", "latency_ms": 1417, "cost_usd": 0.002075, "request_id": "670b3c09-2164-439d-8fc6-a3d1ce46d81e"}
{"ts": "2026-06-30T07:40:03.960553Z", "model": "us.anthropic.claude-haiku-4-5-20251001-v1:0", "input_tokens": 1812, "output_tokens": 235, "stop_reason": "tool_use", "latency_ms": 2394, "cost_usd": 0.002987, "request_id": "0090b4ab-3300-45a0-8c20-e0593d111efe"}
{
  "recommended_flight": "TM488",
  "alternatives": [
    "TM902"
  ],
  "meal_voucher": true,
  "hotel_voucher": true,
  "confidence": 0.95,
  "message_draft": "We're sorry TM482 was cancelled. Your FLEX fare qualifi

## What moved, across the whole build

| Act | Change | Quality | Latency | Cost |
|---|---|---|---|---|
| 1 | working agent | baseline | baseline | baseline |
| 2 | failure recovery, retries, guardrail, structured output | up | flat | slight up |
| 3 | parallel tools, concurrency, two-temperature, streaming | slight up | down | flat |
| 4 | prompt caching, cascade | flat | flat | down |
| 5 | logging, eval harness | decidable | measurable | measurable |
| 6 | single call when the flow is fixed | flat | down | down |

## The durable rules

- An agent is a loop that resends a growing history, so cost grows with the square of the turns. Use the fewest turns that hit your quality bar.
- Tools fetch data or take actions. Writing and reasoning are generation. Do not wrap generation in a tool.
- Keep authoritative rules in code, behind tools, not in the model's head.
- Enforce dangerous boundaries by omitting the tool, not by asking the prompt.
- Cache the fixed prefix, parallelize independent tools, and cascade from cheap to strong on confidence.
- Force structured output for anything a machine consumes, and validate it.
- Log every call, and pick your model with an eval, not a hunch.
- If you already know what to fetch, do not build an agent. Make one structured call.